<!-- NINO26-CABECALHO v1 -->
# 3L — Caracterização das 4 fases: El Niño x La Niña

**Projeto NINO-BRASIL — Oceanografia Física UFPE — Thiago Vilar**  
**Código da fase/letra:** `3L`  ·  **Hipótese:** HIP0

## Descritivo (por que este notebook existe)
Camada visual que percorre os quatro períodos para EN e LN, com duração por classe e discriminantes por fase — o fecho descritivo do mecanismo.

## Pergunta
Como El Niño e La Niña percorrem gênese, crescimento, pico e decaimento, quanto dura cada fase por classe e quais variáveis melhor delimitam cada período?

## Desafio (hipótese a testar)
A assimetria EN/LN precisa aparecer; a caracterização é diagnóstica (rótulos post hoc), não previsão.

## Metodologia (com referências)
Ciclo de vida por fase, duração por classe, heatmap de discriminantes e PCA por fase (acoplamento de Bjerknes, 1969; recarga de Jin, 1997).

## Contrato de saídas — código predecessor único
Cada figura nasce do **mesmo** `registrar_figura(...)` que congela sua numeric-table sob o **mesmo código**, reescrevendo por **sobreposição** a cada execução:

```python
from nino_brasil.viz import registrar_figura
registrar_figura(fig, "Fig_3L01", fase=3, bloco="L",
                 titulo=..., descricao=..., hipotese="HIP0",
                 notebook="notebooks/fase3/3L_en_ln_caracterizacao.ipynb",
                 fontes={"<tabela>": df})   # -> figures/fase3/<codigo>.png + numeric-tables/fase3/<codigo>/
```

| Código | Figura (`figures/fase3/<código>.png`) | Numeric-table (`numeric-tables/fase3/<código>/`) | Descrição |
|---|---|---|---|
| `Fig_3L01` | `Fig_3L01.png` | `Fig_3L01/` | ciclo de vida EN x LN |
| `Fig_3L02` | `Fig_3L02.png` | `Fig_3L02/` | heatmap de discriminantes |
| `Fig_3L03` | `Fig_3L03.png` | `Fig_3L03/` | duração das fases EN x LN |
| `Fig_3L04` | `Fig_3L04.png` | `Fig_3L04/` | PCA por fase |

> Padrão em `docs/PADRAO_NOTEBOOKS.md`; validação por `python scripts/validar_figuras.py --strict`.

In [ ]:
import sys; sys.path.insert(0,'.')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
import fase3_utils as u
STATS, FIGS = u.STATS, u.FIGS
m = u.weekly_matrix()                       # matriz-mestre (31 vars quando completa)
lc = pd.read_csv(STATS/'phase3_fases_semanais_en_ln.csv', parse_dates=['week_ending_sunday']).set_index('week_ending_sunday')
ev = pd.read_csv(STATS/'phase3_events_en_ln.csv', parse_dates=['onset','pico','fim'])
dur = pd.read_csv(STATS/'phase3_duracao_por_tipo_classe.csv')
disc = pd.read_csv(STATS/'phase3_discriminantes_por_periodo.csv')
pcs = pd.read_csv(STATS/'phase3_pca_por_fase.csv')
FASES = ['genese','crescimento','pico','decaimento']
FCOR = {'genese':'#93c5fd','crescimento':'#fca5a5','pico':'#111827','decaimento':'#d1d5db'}
print('master', m.shape, '|', m.index.min().date(),'->',m.index.max().date())
print('eventos: EN', int((ev.tipo=="el_nino").sum()), '| LN', int((ev.tipo=="la_nina").sum()))


In [ ]:
# ---- Fig 1: ciclo de vida composto EN x LN, com as 4 fases rotuladas ----
REL = list(range(-52, 53))
def peak_al(s, peak):
    i = s.index.get_indexer([pd.to_datetime(peak)], method='nearest')[0]
    seg = s.iloc[max(0,i-52):i+53]; rel = ((seg.index - s.index[i]).days/7).round().astype(int)
    return pd.Series(seg.values, index=rel).reindex(REL)
RECARGA = [c for c in ['d20_m','ssh_m','ohc_0_300','wwv','tilt_m'] if c in m.columns]
fig, axes = plt.subplots(2, 2, figsize=(15, 9.6), sharex=True)
for j, tipo in enumerate(['el_nino','la_nina']):
    sub = ev[ev.tipo==tipo]
    nome = 'El Nino' if tipo=='el_nino' else 'La Nina'
    ssta = pd.DataFrame([peak_al(m['nino34_ssta'], e.pico) for _,e in sub.iterrows()]).mean()
    zc = [ (m[v]-m[v].mean())/m[v].std() for v in RECARGA ]
    rec = pd.concat([pd.DataFrame([peak_al(z, e.pico) for _,e in sub.iterrows()]).mean() for z in zc], axis=1).mean(axis=1)
    axes[0,j].plot(REL, ssta.values, color='#b91c1c' if tipo=='el_nino' else '#1d4ed8', lw=2.6,
                   label=f'SSTA composta ({nome}, n={len(sub)})')
    axes[0,j].axhline(0.5 if tipo=='el_nino' else -0.5, color='0.5', ls=':', lw=.8)
    axes[0,j].text(-51, 0.53 if tipo=='el_nino' else -0.62, 'limiar +0.5 C' if tipo=='el_nino' else 'limiar -0.5 C', fontsize=7, color='0.4')
    axes[0,j].set_title(f'{nome} (n={len(sub)}) - SSTA composta alinhada ao pico', fontsize=10, pad=26)
    axes[0,j].set_ylabel('SSTA (C)')
    axes[0,j].legend(fontsize=7.6, loc='lower left')
    axes[1,j].plot(REL, rec.values, color='#111827', lw=2.2,
                   label='recarga = media z de D20, SSH, OHC0-300, WWV e tilt')
    axes[1,j].axhline(0, color='0.5', lw=.5)
    axes[1,j].set_ylabel('recarga (z medio)'); axes[1,j].set_xlabel('semanas rel. ao pico')
    axes[1,j].legend(fontsize=7.6, loc='lower left')
    for ax in (axes[0,j], axes[1,j]):
        ax.axvspan(-52,-26,color=FCOR['genese'],alpha=.25,lw=0); ax.axvspan(-26,0,color=FCOR['crescimento'],alpha=.25,lw=0)
        ax.axvspan(0,26,color=FCOR['decaimento'],alpha=.3,lw=0); ax.axvline(0,color='k',ls='--',lw=1); ax.grid(alpha=.2)
    trans = axes[0,j].get_xaxis_transform()
    axes[0,j].text(-39, 1.03, 'I. GENESE', transform=trans, ha='center', fontsize=8.4, color='#1e3a8a', weight='bold')
    axes[0,j].text(-13, 1.03, 'II. CRESCIMENTO', transform=trans, ha='center', fontsize=8.4, color='#7f1d1d', weight='bold')
    axes[0,j].text(0, 1.10, 'III. PICO', transform=trans, ha='center', fontsize=8.4, color='k', weight='bold')
    axes[0,j].text(14, 1.03, 'IV. DECAIMENTO', transform=trans, ha='center', fontsize=8.4, color='#374151', weight='bold')
fig.suptitle('3L - Ciclo de vida composto por sinal: El Nino x La Nina (4 fases mapeadas; recarga espelhada)', fontsize=13)
u.add_note(axes[1,1], 'Leitura: no El Nino a recarga (calor subsuperficial) sobe ANTES do pico e descarrega depois;\nna La Nina o espelho: descarga antes do pico frio e recarga na saida.\nFaixas: azul=genese, rosa=crescimento, cinza=decaimento; linha tracejada=pico.', loc='lower right')
u.stamp_caption(fig, variavel='SSTA e recarga compostas alinhadas ao pico', area='Nino 3.4', periodo='eventos 1981-2026', fonte='matriz-mestre semanal', extra='faixas: genese/crescimento/pico/decaimento; tabelas phase3_*_en_ln.csv')
u.save_fig(fig, 'phase3L_ciclo_vida_en_ln.png')
plt.show()


In [ ]:
# ---- Fig 2: duracao por fase com valores ----
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), sharey=True)
for ax, tipo in zip(axes, ['el_nino','la_nina']):
    d = dur[dur.tipo==tipo]
    classes = list(dict.fromkeys(d.classe))
    x = np.arange(len(FASES)); wdt = 0.8/max(len(classes),1)
    for k, cl in enumerate(classes):
        vals = [float(d[(d.classe==cl)&(d.fase==f)]['duracao_media_semanas'].mean() or 0) for f in FASES]
        bars = ax.bar(x + k*wdt, vals, width=wdt, label=cl)
        for b, v in zip(bars, vals):
            if v > 0:
                ax.text(b.get_x()+b.get_width()/2, v+0.5, f'{v:.0f}', ha='center', fontsize=6.6)
    ax.set_xticks(x + wdt*(len(classes)-1)/2); ax.set_xticklabels(['I. genese','II. crescimento','III. pico','IV. decaimento'], fontsize=9)
    ax.set_title(f'{"El Nino" if tipo=="el_nino" else "La Nina"}: duracao media de cada fase por classe (semanas)', fontsize=10)
    ax.legend(fontsize=8, title='classe'); ax.grid(axis='y', alpha=.25)
axes[0].set_ylabel('duracao media (semanas)')
fig.suptitle('3L - Quanto dura cada fase do ciclo de vida, por intensidade do evento')
u.stamp_caption(fig, variavel='duracao media de cada fase por classe', area='Nino 3.4', periodo='1981-2026', fonte='phase3_duracao_por_tipo_classe.csv', extra='numeros sobre as barras = semanas')
u.save_fig(fig, 'phase3L_duracao_fases_en_ln.png')
plt.show()


In [ ]:
# ---- Fig 3: heatmap discriminante ----
piv = disc.pivot(index='variavel', columns='tipo', values='epsilon2_entre_fases')
piv = piv.reindex(piv.mean(axis=1).sort_values(ascending=False).index)
fig, ax = plt.subplots(figsize=(8.2, max(6, .32*len(piv))))
im = ax.imshow(piv.values, cmap='magma_r', aspect='auto', vmin=0, vmax=float(np.nanmax(piv.values)))
ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(['El Nino' if c=='el_nino' else 'La Nina' for c in piv.columns], fontsize=9)
ax.set_yticks(range(len(piv))); ax.set_yticklabels(piv.index, fontsize=8)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v = piv.values[i,j]
        if np.isfinite(v): ax.text(j,i,f'{v:.2f}',ha='center',va='center',fontsize=7,color='white' if v>0.5*np.nanmax(piv.values) else 'black')
ax.set_title('3L - Quais variaveis melhor separam as 4 fases do ciclo?\n(epsilon^2 de Kruskal-Wallis entre genese/crescimento/pico/decaimento)', fontsize=11)
fig.colorbar(im, ax=ax, shrink=.7, label='poder discriminante (epsilon^2: 0=nenhum, 1=total)')
ax.text(1.02, -0.03, 'Leitura: valores altos (escuros) = a variavel muda de nivel entre as fases\ne por isso ajuda a dizer em que fase o evento esta. SSTA, tilt, T100m e\nOHC discriminam bem o El Nino; na La Nina o sinal fica mais raso (T50m/OHC0-100).',
        transform=ax.transAxes, fontsize=7.6, ha='right', va='top')
u.stamp_caption(fig, variavel='epsilon^2 entre as 4 fases por variavel', area='Nino 3.4', periodo='1981-2026 semanal', fonte='phase3_discriminantes_por_periodo.csv')
u.save_fig(fig, 'phase3L_discriminantes_heatmap.png')
plt.show()


In [ ]:
# ---- Fig 4: PCA por fase ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5.0), sharey=True)
for ax, tipo in zip(axes, ['el_nino','la_nina']):
    sub = pcs[pcs.tipo==tipo]
    x = np.arange(len(FASES)); wdt=0.2
    for k in range(1,4):
        pc = sub[sub.componente==f'PC{k}']
        vals = [float(pc[pc.fase==f]['var_explicada'].mean() or 0) for f in FASES]
        bars = ax.bar(x + (k-2)*wdt, vals, width=wdt, label=f'PC{k}')
        for b, v in zip(bars, vals):
            if v > 0: ax.text(b.get_x()+b.get_width()/2, v+0.008, f'{v*100:.0f}%', ha='center', fontsize=6.4)
    ax.set_xticks(x); ax.set_xticklabels(['I. genese','II. crescimento','III. pico','IV. decaimento'], fontsize=9)
    ax.set_title(f'{"El Nino" if tipo=="el_nino" else "La Nina"}: fracao da variancia explicada por PC, em cada fase', fontsize=10)
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=.25)
axes[0].set_ylabel('fracao da variancia')
fig.suptitle('3L - Complexidade de cada fase: quantas dimensoes fisicas independentes o periodo exige')
u.add_note(axes[1], 'Leitura: PC1 alto = fase dominada por UMA dimensao fisica (ex.: recarga no crescimento);\nPC1 baixo com PC2/PC3 relevantes = fase mais complexa, com processos concorrentes.', loc='upper right')
u.stamp_caption(fig, variavel='variancia explicada PCA por fase', area='Nino 3.4', periodo='1981-2026', fonte='phase3_pca_por_fase.csv', extra='PCA/EOF por ciclo de vida; numeros = % da variancia')
u.save_fig(fig, 'phase3L_pca_por_fase.png')
plt.show()


**Leitura do 3L.** As figuras fecham a camada visual da Fase 3 para os dois sinais, com as 4 fases (I. genese, II. crescimento, III. pico, IV. decaimento) rotuladas em todos os paineis: o ciclo de vida composto mostra a recarga liderando o aquecimento no El Nino e a descarga espelhada na La Nina; a duracao por fase (valores em semanas nas barras) separa fracos de fortes/super; o heatmap identifica as variaveis que melhor delimitam cada periodo (SSTA, tilt e recarga subsuperficial no topo do El Nino; T50m/OHC0-100 na La Nina); e a PCA por fase quantifica a complexidade de cada periodo (PC1 alto = fase dominada por uma unica dimensao fisica). Todos os numeros vem das tabelas `phase3_*_en_ln.csv`.


<!-- NINO26-REFERENCIAS v1 -->
## Referências Bibliográficas

1. Bjerknes, J. (1969). Atmospheric Teleconnections from the Equatorial Pacific. *Monthly Weather Review*, 97(3), 163-172. https://doi.org/10.1175/1520-0493(1969)097<0163:ATFTEP>2.3.CO;2
2. Jin, F.-F. (1997). An Equatorial Ocean Recharge Paradigm for ENSO. Part I. *J. Atmos. Sci.*, 54, 811-829. https://doi.org/10.1175/1520-0469(1997)054<0811:AEORPF>2.0.CO;2

Relação completa em `Artigos_Referências/Referências_Bibliográficas.xls`.